In [11]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [12]:
# 1. Define state schema
class ChatbotState(TypedDict):
    messages: Annotated[list, add_messages]

In [13]:
# 2. Define external tool (AI News)
def ai_news_tool(state: ChatbotState):
    # Normally you'd call an external API here.
    # For demo, we'll simulate with a static response.
    reply = "Latest AI News: Anthropic adds invisible watermarks, Alibaba Qwen surpasses 3B downloads, AI predicted to cure diseases in 5–10 years."
    return {"messages": [{"role": "assistant", "content": reply}]}

In [14]:
# 3. Define chatbot node
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")

In [15]:
def chatbot_node(state: ChatbotState):
    last_message = state["messages"][-1].content if state["messages"] else ""
    if "news" in last_message.lower():
        return ai_news_tool(state)
    else:
        reply = llm.invoke(state["messages"])
        return {"messages": [reply]}

In [16]:
# 4. Build graph
graph_builder = StateGraph(ChatbotState)
graph_builder.add_node("llmchatbot", chatbot_node)
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

In [17]:
# 5. Compile graph
graph = graph_builder.compile()

In [18]:
# 6. Run chatbot
response = graph.invoke({"messages": [{"role": "user", "content": "Hello!"}]})
print(response["messages"][-1].content)

Hello! How can I assist you today?


In [19]:
response = graph.invoke({"messages": [{"role": "user", "content": "Give me AI news"}]})
print(response["messages"][-1].content)

Latest AI News: Anthropic adds invisible watermarks, Alibaba Qwen surpasses 3B downloads, AI predicted to cure diseases in 5–10 years.
